# FADE · GENERIC arm, rebuilt (D39 + D40)

The control for `fade_typed.ipynb`. Same 1,500 problems, same forked pass-1, same
exemplar budget, same retry budget. **Run them side by side.**

### What generic gets, and what it must never get

| | typed | generic |
|---|---|---|
| retrieval by question | yes | yes (see the switch below) |
| `ftype` passed to retrieval | **yes** | no |
| type-fit re-rank | **yes** | no |
| typed instruction | **yes** | none — asserted |
| failure-mode warning | optional | never |

Everything type-specific stays out. There is an `assert not instruction` on the generic
branch of `retry_once` so it cannot drift.

### The one switch that decides what you are measuring

`config.GENERIC_QUESTION_AWARE`

- **`True` (default)** — both arms retrieve by question, so they differ *only* by
  typing. A typed win is attributable to **typing**. This is the comparison the paper
  needs.
- **`False`** — generic stays question-blind (last k of the pool). A typed win then
  confounds typing with retrieval.

Your completed pre-fix run is **already** the question-blind comparison — typed 40.3%
vs generic 39.3%, McNemar p=0.374. So leaving this `True` gives you both comparisons;
setting it `False` re-runs an arm you already have.

### Names

Everything carries a `_d40` suffix. Reusing the old names would make `fetch_store`
restore the completed pre-fix store and skip the run silently.

~5 GPU-h. Sidebar: GPU · Internet · Secrets `HF_TOKEN`, `GH_TOKEN`.

### 1 · Setup

In [1]:
import os, sys, shutil, subprocess, json, datetime
!pip -q install -U "transformers>=4.40" accelerate sentence-transformers sympy scikit-learn datasets scipy 2>/dev/null | tail -1
REPO='/kaggle/working/fade'; STORES='/kaggle/working/stores'; os.makedirs(STORES,exist_ok=True)
if os.path.exists(REPO): shutil.rmtree(REPO)
!git clone --depth 1 https://github.com/S2V3/fade.git {REPO}
sys.path.insert(0,REPO)
import importlib, config; importlib.reload(config)
VER=config.FADE_CODE_VERSION; assert VER=='audit-2', f'STALE CODE {VER!r}'
for fn in ('autopush.py','fork_pass1_store.py','check_arm_separation.py','compare_arms.py'):
    assert os.path.exists(f'{REPO}/{fn}'), f'{fn} missing -- push it to the repo'
from kaggle_secrets import UserSecretsClient
_us=UserSecretsClient(); HF=_us.get_secret('HF_TOKEN'); GH=_us.get_secret('GH_TOKEN')
os.environ['FADE_HF_TOKEN']=HF
MODEL='meta-llama/Llama-2-7b-chat-hf'; TAG=VER.replace('-','')
print('code',VER,'OK')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.5 MB/s eta 0:00:00
Cloning into '/kaggle/working/fade'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 42 (delta 0), reused 32 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 1.36 MiB | 7.41 MiB/s, done.
code audit-2 OK


### 2 · Guard, and set the switch

Must match `fade_typed.ipynb` exactly, or the arms are not comparable.

In [2]:
import inspect
import kaggle_run as K, config

GENERIC_QUESTION_AWARE = True     # <- the decision. See the note at the top.
config.GENERIC_QUESTION_AWARE = GENERIC_QUESTION_AWARE

# COMPUTE PARITY with typed's D44 acceptance test. Typed resamples the ~23% of
# retries that fail their type's gold-free test; generic has no test, so it
# resamples the same FRACTION at random and keeps the second draw. Without this
# typed gets 23% more generations and the win is attributable to compute.
config.GENERIC_MATCH_RESAMPLE = True
config.GENERIC_RESAMPLE_RATE  = 0.231   # measured from the pre-fix typed run
print(f'generic matched resample rate: {config.GENERIC_RESAMPLE_RATE:.1%}')

sig = inspect.signature(K.select_generic_positives).parameters
assert 'question' in sig, "kaggle_run.py is the OLD question-blind version (D39)"
assert hasattr(K,'_retrieve'), "kaggle_run.py is missing _retrieve (D39/D40)"
for k in ('RETRY_QUESTION_AWARE','GENERIC_QUESTION_AWARE','RETRY_W_TYPEFIT',
          'TYPED_INSTRUCTION_STYLE','TYPED_APPROACH_DEMO','RETRY_RELEVANCE_LAST',
          'GENERIC_MATCH_RESAMPLE','GENERIC_RESAMPLE_RATE'):
    assert hasattr(config,k), f'config.py is missing {k}'
h = subprocess.run([sys.executable,'kaggle_run.py','--help'], cwd=REPO,
                   capture_output=True, text=True).stdout
assert '--store-dir' in h, 'kaggle_run.py predates --store-dir (D38)'

print(f'RETRY_QUESTION_AWARE    {config.RETRY_QUESTION_AWARE}')
print(f'GENERIC_QUESTION_AWARE  {config.GENERIC_QUESTION_AWARE}')
print(f'  -> generic ' + ('retrieves by question; a typed win is attributable to TYPING'
                          if GENERIC_QUESTION_AWARE else
                          'is question-BLIND; a typed win confounds typing with retrieval'))
print('\n  guard passed')

Torch OK
 sentence-transformers OK
generic matched resample rate: 23.1%
RETRY_QUESTION_AWARE    True
GENERIC_QUESTION_AWARE  True
  -> generic retrieves by question; a typed win is attributable to TYPING

  guard passed


In [3]:
# --- github sync ------------------------------------------------------
AUTH=f'https://{GH}@github.com/S2V3/fade.git'; BRANCH='results'
def sh(cmd):
    p=subprocess.Popen(cmd,shell=True,cwd=REPO,stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT,text=True,bufsize=1)
    for ln in p.stdout: print(ln,end='',flush=True)
    p.wait(); return p.returncode
def _rows(p, phase=None):
    f=os.path.join(p,'results.jsonl')
    if not os.path.exists(f): return 0
    if phase is None: return sum(1 for _ in open(f))
    n=0
    for ln in open(f):
        if not ln.strip(): continue
        try:
            if json.loads(ln).get('phase')==phase: n+=1
        except Exception: pass
    return n
def _ver(p):
    try: return json.load(open(os.path.join(p,'config_snapshot.json'))).get('FADE_CODE_VERSION')
    except Exception: return None
def fetch_store(run,dest):
    w=f'/kaggle/working/_rs_{run}'; c=[]; shutil.rmtree(w,ignore_errors=True)
    if subprocess.run(['git','clone','--depth','1','--branch',BRANCH,AUTH,w],
                      capture_output=True,text=True).returncode==0 and os.path.isdir(f'{w}/results'):
        for d in sorted(os.listdir(f'{w}/results')):
            if d.startswith(f'store_{run}_'):
                full=f'{w}/results/{d}'
                if os.path.isdir(full) and _ver(full)==VER and _rows(full)>0:
                    c.append((_rows(full),d,full))
    c.sort(reverse=True)
    print(f'  [{run}] local {_rows(dest)} | best remote {c[0][0] if c else 0}'
          + (f' ({c[0][1]})' if c else ''))
    if c and c[0][0]>_rows(dest):
        shutil.rmtree(dest,ignore_errors=True); shutil.copytree(c[0][2],dest)
        print(f'  RESTORED {c[0][0]} rows')
    return _rows(dest)
print('sync ready')

sync ready


### 3 · Fork pass-1 from the original run

Identical fork to the typed notebook: the same 1,500 first attempts, the same 255-entry
post-pass-1 pool. Both arms therefore retry the *same* failures from the *same*
starting state.

In [4]:
SRC_RUN = f'typed_train_{TAG}'              # the original run, for its pass-1
CTRL_RUN= f'generic_train_{TAG}_d40'        # NEW name -- never reuse the old one
SRC = f'{STORES}/store_{SRC_RUN}'
CTRL= f'{STORES}/store_{CTRL_RUN}'

fetch_store(SRC_RUN, SRC)
assert _rows(SRC,'pass1')>0, 'no original store on the results branch'
print(f"  source: {_rows(SRC,'pass1')} pass-1 rows")

fetch_store(CTRL_RUN, CTRL)
if _rows(CTRL)==0:
    assert sh(f'python fork_pass1_store.py --src {SRC} --dst {CTRL} --force')==0, 'fork failed'
else:
    print(f"  RESUMING: {_rows(CTRL,'retry')} retries already done")

  [typed_train_audit2] local 0 | best remote 2568 (store_typed_train_audit2_inprogress)
  RESTORED 2568 rows
  source: 1500 pass-1 rows
  [generic_train_audit2_d40] local 0 | best remote 0
  FORKED  /kaggle/working/stores/store_typed_train_audit2  ->  /kaggle/working/stores/store_generic_train_audit2_d40
  pass-1 rows kept          1500
  retry rows dropped        1068
  pool  334 -> 255   (79 were added during retry, removed)
  medium queue              397   {'MEDIUM_WRONG': 220, 'MEDIUM_CORRECT': 177}
  bad queue                 848   {'BAD_WRONG': 848}

  resume will report        1500 problems already scored
  -> pass-1 SKIPPED, 1068 problems go to fresh retries
  note: pool.jsonl carries gold on disk (by design) -- TraceStore.exemplars()
        strips it before anything reaches a prompt.

  next:
    python kaggle_run.py --retry-mode generic --store-dir /kaggle/working/stores/store_generic_train_audit2_d40 \
        --run-name <name> --n-problems 1500 --model <model>


### 4 · Confirm the arms are separated (no GPU)

In [5]:
from pathlib import Path
from classification import TraceStore
from diagnosis import FailureType
pool = TraceStore(root=Path(CTRL)).exemplars()
rows = [json.loads(l) for l in open(f'{CTRL}/results.jsonl') if l.strip()]
wrong= [r for r in rows if r.get('phase')=='pass1'
        and not (r.get('signals') or {}).get('correct')]

sets=set(); ov=[]
for r in wrong[:40]:
    ft = FailureType(r.get('diagnosis') or 'UNCLASSIFIED')
    g = {x.get('question') for x in K.select_generic_positives(pool,[],8,question=r['question'])}
    sets.add(tuple(sorted(g)))
    if ft is not FailureType.UNCLASSIFIED:
        t = {x.get('question') for x in K.select_typed_positives(pool,ft,[],8,question=r['question'])}
        ov.append(len(t & g))
import statistics
print(f'  distinct GENERIC exemplar sets over 40 questions: {len(sets)}'
      f"   ({'question-aware' if len(sets)>1 else 'question-blind'})")
if ov:
    print(f'  mean typed/generic overlap: {statistics.mean(ov):.2f} / 8')
assert sh(f'python check_arm_separation.py --store {CTRL}')==0, 'arms not separated' 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  distinct GENERIC exemplar sets over 40 questions: 40   (question-aware)
  mean typed/generic overlap: 3.53 / 8
Torch OK
 sentence-transformers OK
pool: 255 gold-stripped exemplars


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2554.25it/s]
probe question: Albert is wondering how much pizza he can eat in one day. He buys 2 la...
(overlap is now measured for ONE question; typed and generic share
 relevant exemplars by design, so a non-zero overlap is expected --
 what matters is that it is well below the full budget.)

  1 · TYPED vs GENERIC  (exemplar overlap out of 8)
  type      overlap   instruction   verdict
  NR              4           yes   differs
  AL              2           yes   differs
  ST              3           yes   differs
  SM              2           yes   differs
  CE              2           yes   differs
  WP              3           yes   differs

  generic instruction: (empty)  <- by design

  2 · TYPED vs TYPED  (do the eight cures differ from eac

### 5 · Run (~5 GPU-h, checkpointed every 10 min)

Safe to run at the same time as the typed notebook — different stores, different run
names. Both autopush threads push to the same branch, so an occasional
`push ... FAILED` line is normal: it pulls, rebases and retries on the next cycle.

In [ ]:
import autopush
MAX_TOK = 400      # MUST match the typed notebook -- 320 truncated 7% of retries
before=_rows(CTRL,'retry'); rc=1
ap = autopush.start(CTRL, CTRL_RUN, GH, minutes=10)
try:
    rc = sh(f'python -u kaggle_run.py --retry-mode generic --store-dir {CTRL} '
            f'--run-name {CTRL_RUN} --n-problems 1500 --model {MODEL} '
            f'--max-new-tokens {MAX_TOK} --show-every 25 --secret-name HF_TOKEN')
finally:
    ap.stop()
after=_rows(CTRL,'retry')
print(f'\n  retry rows {before} -> {after}  (exit {rc})')
assert rc==0, f'kaggle_run.py FAILED with exit {rc}'
assert after>0, 'ZERO retries produced -- do not compare'
print('  done and pushed')

  [autopush generic_train_audit2_d40] started, every 10 min -> /kaggle/working/stores/store_generic_train_audit2_d40
Torch OK
 sentence-transformers OK
  generation module version: v5-hashline
  v5-hashline ACTIVE | N_SHOTS=8 rep_penalty=1.15 no_repeat_ngram=0
  ban_strings=['\\begin{code}', '\\end{code}', '```'] | stop_markers=['\nQuestion:', '\nQ:', '\nExample', '\nProblem:']
  preprocessing: normalize_traces=True
  exemplar budget: 8 (enforced across all arms)
  model: meta-llama/Llama-2-7b-chat-hf
  HF token loaded from Kaggle Secret 'HF_TOKEN'
  HF identity: S2V3 (type=user)
  gated-access probe OK: meta-llama/Llama-2-7b-chat-hf reachable (16 files)

Loading GSM8K (train split)...

Generating train split: 100%|██████████| 7473/7473 [00:00<00:00, 113807.39 examples/s]

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 219352.40 examples/s]
  source: openai/gsm8k [train]
  train: 3600 problems ready (preprocessed) | dropped 0 unparseable

Selecting seeds from the first

### 6 · Compare, once both arms have finished

In [ ]:
TYPED_RUN=f'typed_train_{TAG}_d40'
TYPED=f'{STORES}/store_{TYPED_RUN}'
if fetch_store(TYPED_RUN, TYPED) and _rows(TYPED,'retry')>0:
    assert sh(f'python compare_arms.py --typed {TYPED} --generic {CTRL}')==0
    print("""
  typed > generic   the DIAGNOSIS earns the gain -- and with GENEeRIC_QUESTION_AWARE
                    True that is TYPING, not retrieval. This is the paper's claim.
  typed ~= generic  typing adds nothing beyond retrieval + retry. Report deferred
                    retry as the contribution and the taxonomy as instrumentation.
  typed < generic   the type-fit re-rank is costing more relevance than the typing
                    buys. Lower config.RETRY_W_TYPEFIT and reconsider.
""")
else:
    print('\n  typed_d40 has not finished yet. Re-run this cell when it has.')
    print('  Pre-fix reference (question-blind both sides):')
    print('    typed 40.3%  generic 39.3%  |  114 vs 100  p = 0.374')